In [1]:
import os 
cwd = os.getcwd()
cwd

os.chdir("/Users/jmthomas565/Desktop/Education/Applied Stats MSc/MSc Project/Git Files/msc-project-source-code-files-24-25-jmthomas565/Code/Project Code Pipeline")

new_cwd = os.getcwd() 

new_cwd


'/Users/jmthomas565/Desktop/Education/Applied Stats MSc/MSc Project/Git Files/msc-project-source-code-files-24-25-jmthomas565/Code/Project Code Pipeline'

In [2]:
%run 00_inititial_transformations.ipynb



['2017-07-01', '2017-07-02', '2017-07-03', '2017-07-04', '2017-07-05'] ... ['2022-09-27', '2022-09-28', '2022-09-29', '2022-09-30', '2022-10-01']


In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import warnings

In [4]:
# Function to return Adstocked variables
def adstock(x, theta):
    return tsa.filters.recursive_filter(x, theta)

def saturation(x, K, S):
    return x**S / (K**S + x**S)

In [5]:


# Locking in final AdStock Scores
# Note that offline media doesn't receive any adstock, because of the previous optimisation

X_train["branded_search_spend_adstock"] = adstock(X_train["branded_search_spend"], [0.9])

X_train["nonbranded_search_spend_adstock"] = adstock(X_train["nonbranded_search_spend"], [0.9])

X_train["facebook_spend_adstock"] = adstock(X_train["facebook_spend"], [0.9])

X_train["print_spend_adstock"] = X_train["print_spend"]

X_train["ooh_spend_adstock"] = X_train["ooh_spend"]

X_train["tv_spend_adstock"] = X_train["tv_spend"]

X_train["radio_spend_adstock"] = X_train["radio_spend"]

### Online Saturation

In [7]:

#Cross-validation hold-out
hold_out_split = -26
y_train_calib = y_train.iloc[:hold_out_split]
y_train_hold_out = y_train.iloc[hold_out_split:]


def adj_r2(y_true, y_pred, p):
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    #Making sure the denominator of adj R^2 > 0
    if n - p - 1 <= 0:
        return np.nan
    calculation = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return calculation

In [8]:
#Testing online channels across the saturation parameter space, whilst holding offline channels fixed

def test_saturation_4var_model(X_raw, K_range, s_range):

    #setting up the 3 online and 1 offline columns (all with adstock already added)
    sat_cols = ["branded_search_spend_adstock", "nonbranded_search_spend_adstock", "facebook_spend_adstock"]
    offline_col = 'print_spend_adstock'

    for K in K_range:
        for s in s_range:
            #creates a copy of X_raw to avoid a cummulitive saturation effect
            X_copy = X_raw.copy()

            #saturation is applied to the copied column
            for col in sat_cols:
                X_copy[f"{col}_sat"] = saturation(X_copy[col], K, s)

            all_features = [f"{col}_sat" for col in sat_cols] + [offline_col]
            X_model = X_copy[all_features]


            #splitting for hold-out cv
            X_train_calib = X_model.iloc[:hold_out_split]
            X_val_hold_out = X_model.iloc[hold_out_split:]

            #standardising the data
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_calib)
            X_val_scaled = scaler.transform(X_val_hold_out)

            #training the model
            ridge = Ridge(alpha=1.0, positive=True)
            ridge.fit(X_train_scaled, y_train_calib)
            y_pred = ridge.predict(X_val_scaled)

            #prinintg the R^2_bar values
            score = adj_r2(y_train_hold_out, y_pred, p=X_train_scaled.shape[1])
            print(f"K: {round(K, 2)}, s: {round(s,2)}: Adjusted R2 = {round(score, 4)}")

# Define the ranges for K and s
K_range = np.arange(0.1, 1.0, 0.1)
s_range = np.arange(0.1, 1.0, 0.1)

# Now call the refined function
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    test_saturation_4var_model(X_train, K_range, s_range)

K: 0.1, s: 0.1: Adjusted R2 = 0.0047
K: 0.1, s: 0.2: Adjusted R2 = -0.1031
K: 0.1, s: 0.3: Adjusted R2 = -0.1998
K: 0.1, s: 0.4: Adjusted R2 = -0.2746
K: 0.1, s: 0.5: Adjusted R2 = -0.3647
K: 0.1, s: 0.6: Adjusted R2 = -0.4313
K: 0.1, s: 0.7: Adjusted R2 = -0.4594
K: 0.1, s: 0.8: Adjusted R2 = -0.4198
K: 0.1, s: 0.9: Adjusted R2 = -0.4926
K: 0.2, s: 0.1: Adjusted R2 = 0.0067
K: 0.2, s: 0.2: Adjusted R2 = -0.1
K: 0.2, s: 0.3: Adjusted R2 = -0.1968
K: 0.2, s: 0.4: Adjusted R2 = -0.2723
K: 0.2, s: 0.5: Adjusted R2 = -0.3654
K: 0.2, s: 0.6: Adjusted R2 = -0.4307
K: 0.2, s: 0.7: Adjusted R2 = -0.4592
K: 0.2, s: 0.8: Adjusted R2 = -0.4197
K: 0.2, s: 0.9: Adjusted R2 = -0.4925
K: 0.3, s: 0.1: Adjusted R2 = 0.0079
K: 0.3, s: 0.2: Adjusted R2 = -0.0979
K: 0.3, s: 0.3: Adjusted R2 = -0.1948
K: 0.3, s: 0.4: Adjusted R2 = -0.2706
K: 0.3, s: 0.5: Adjusted R2 = -0.3644
K: 0.3, s: 0.6: Adjusted R2 = -0.4302
K: 0.3, s: 0.7: Adjusted R2 = -0.4591
K: 0.3, s: 0.8: Adjusted R2 = -0.4196
K: 0.3, s: 0.9: Ad

### Offline Saturation

In [10]:
#Testing offline channels across the saturation parameter space, whilst holding online channels fixed

def test_saturation_4var_model(X_raw, K_range, s_range):

    #setting up the 3 online and 1 offline columns (all with adstock already added)
    sat_cols =  ['print_spend']
    online_col = ["branded_search_spend_adstock", "nonbranded_search_spend_adstock", "facebook_spend_adstock"]

    for K in K_range:
        for s in s_range:
            #creates a copy of X_raw to avoid a cummulitive saturation effect
            X_copy = X_raw.copy()

            #saturation is applied to the copied column
            for col in sat_cols:
                X_copy[f"{col}_sat"] = saturation(X_copy[col], K, s)

            all_features = [f"{col}_sat" for col in sat_cols] + online_col
            X_model = X_copy[all_features]


            #splitting for hold-out cv
            X_train_calib = X_model.iloc[:hold_out_split]
            X_val_hold_out = X_model.iloc[hold_out_split:]

            #standardising the data
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_calib)
            X_val_scaled = scaler.transform(X_val_hold_out)

            #training the model
            ridge = Ridge(alpha=1.0, positive=True)
            ridge.fit(X_train_scaled, y_train_calib)
            y_pred = ridge.predict(X_val_scaled)

            #prinintg the R^2_bar values
            score = adj_r2(y_train_hold_out, y_pred, p=X_train_scaled.shape[1])
            print(f"K: {round(K, 2)}, s: {round(s,2)}: Adjusted R2 = {round(score, 4)}")

#defining the s and K ranges
K_range = np.arange(0.1, 1.0, 0.1)
s_range = np.arange(0.1, 1.0, 0.1)

#printing the final function
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    test_saturation_4var_model(X_train, K_range, s_range)

K: 0.1, s: 0.1: Adjusted R2 = 0.4102
K: 0.1, s: 0.2: Adjusted R2 = 0.4102
K: 0.1, s: 0.3: Adjusted R2 = 0.4102
K: 0.1, s: 0.4: Adjusted R2 = 0.4102
K: 0.1, s: 0.5: Adjusted R2 = 0.4102
K: 0.1, s: 0.6: Adjusted R2 = 0.4102
K: 0.1, s: 0.7: Adjusted R2 = 0.4102
K: 0.1, s: 0.8: Adjusted R2 = 0.4102
K: 0.1, s: 0.9: Adjusted R2 = 0.4102
K: 0.2, s: 0.1: Adjusted R2 = 0.4102
K: 0.2, s: 0.2: Adjusted R2 = 0.4102
K: 0.2, s: 0.3: Adjusted R2 = 0.4102
K: 0.2, s: 0.4: Adjusted R2 = 0.4102
K: 0.2, s: 0.5: Adjusted R2 = 0.4102
K: 0.2, s: 0.6: Adjusted R2 = 0.4102
K: 0.2, s: 0.7: Adjusted R2 = 0.4102
K: 0.2, s: 0.8: Adjusted R2 = 0.4102
K: 0.2, s: 0.9: Adjusted R2 = 0.4102
K: 0.3, s: 0.1: Adjusted R2 = 0.4102
K: 0.3, s: 0.2: Adjusted R2 = 0.4102
K: 0.3, s: 0.3: Adjusted R2 = 0.4102
K: 0.3, s: 0.4: Adjusted R2 = 0.4102
K: 0.3, s: 0.5: Adjusted R2 = 0.4102
K: 0.3, s: 0.6: Adjusted R2 = 0.4102
K: 0.3, s: 0.7: Adjusted R2 = 0.4102
K: 0.3, s: 0.8: Adjusted R2 = 0.4102
K: 0.3, s: 0.9: Adjusted R2 = 0.4102
K